In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name: str):
    # ツール定義（検索）
    tools = [TavilySearchResults(max_results=5)]

    # LLM
    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)

    graph_builder = StateGraph(State)

    # チャットボットノード
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder.add_node("chatbot", chatbot)

    # ツールノード
    tool_node = ToolNode(tools)
    graph_builder.add_node("tools", tool_node)

    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )

    # tools → chatbot（ツール結果を踏まえて次の応答）
    graph_builder.add_edge("tools", "chatbot")

    # エントリポイント
    graph_builder.set_entry_point("chatbot")

    # メモリ付きでコンパイル
    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)

    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values",
    )
    for event in events:
        message = event["messages"][-1]

        # assistant のメッセージだけ表示
        if message.type == "ai":
            print(message.content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ["OPENAI_API_KEY"] = os.environ["API_KEY"]

# モデル名
MODEL_NAME = "gpt-4o-mini"

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
print("こんにちは！")
while True:
    user_input = input("質問:")
    if user_input.strip() == "":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

/Users/tera/Downloads/llmdev/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


こんにちは！
こんにちは
こんにちは！今日はどんなことをお手伝いできますか？
今日の天気を教えて

[{"url": "https://weather.yahoo.co.jp/weather/", "content": "2月20日（金）【熊本の天気】3連休は春の陽気 日曜は春一番の可能性も　RKK気象予報士の天気解説 ＜阿蘇や天草のライブカメラも配信中＞\n3連休の天気　21日は行楽日和　23日は北海道を中心に荒天　気温上昇　花粉に注意\n【なだれ注意報】石川県・金沢市、小松市、加賀市、白山市に発表（雪崩注意報） 20日10:02時点\n【なだれ注意報】山形県・山形市、米沢市、鶴岡市、酒田市、新庄市、寒河江市などに発表（雪崩注意報） 20日09:55時点\n【近畿の天気】20日（金）朝　大阪は曇りのち晴れ　最高気温は19日と同じくらいのところが多い予想\n2025年には過去最遅の記録も 北海道に届く流氷はどこからやってくる？【眠れなくなるほど面白い 図解 北海道の話】\n【青森の天気】20日（金）青森・弘前は曇り　八戸は最高8℃の予想\n「スギ花粉」の飛散状況をチェック！早めの対策が重要だけど…気になる県内の状況は？ 神戸市など\n小さな春　富山県内でフキノトウ\n九州・沖縄（福岡、佐賀、長崎、熊本、大分、宮崎、鹿児島、沖縄）　今日（２０日）の天気\n四国（徳島、香川、愛媛、高知）　今日（２０日）の天気\n中国（岡山、広島、山口、鳥取、島根）　今日（２０日）の天気\n黄砂が22日～23日に西日本に飛来か　3連休は黄砂と花粉のダブルパンチに注意\n関西（滋賀、京都、大阪、兵庫、奈良、和歌山）　今日（２０日）の天気\n東海（静岡、愛知、岐阜、三重）　今日（２０日）の天気\n甲信越・北陸（山梨、長野、新潟、富山、石川、福井）　今日（２０日）の天気\n東京　今日（２０日）の天気\n【きょう2/20（金）広島天気】午後は次第に晴れる　気温上昇で日差しの暖かさも\n関東（茨城、栃木、群馬、埼玉、千葉、神奈川）　今日（２０日）の天気\n東北（青森、岩手、宮城、秋田、山形、福島）　今日（２０日）の天気\n北海道　今日（２０日）の天気 [...] 東北（青森、岩手、宮城、秋田、山形、福島）　今日（２０日）の天気\n北海道　今日（２０日）の天気\n【天気】太平洋側は雲多め　にわか雨も